In [44]:
import pandas as pd

## Merging DLI

In [72]:
df1 = pd.read_csv('DLI_1.csv')
df1.columns

Index(['brand', 'price', 'start_time', 'start_day', 'end_day', 'end_time',
       'trip_time', 'take_place', 'destination', 'crawl_date',
       'checked_baggage', 'hand_luggage'],
      dtype='object')

In [73]:
city = 'DLI'
df2 = pd.read_csv('DLI_2.csv')
df2.columns

Index(['brand', 'price', 'start_time', 'start_day', 'end_time', 'end_day',
       'trip_time', 'take_place', 'destination', 'hand_luggage',
       'checked_baggage', 'crawl_date'],
      dtype='object')

In [74]:
order = ['brand', 'price', 'start_time', 'start_day', 'end_time', 'end_day',
       'trip_time', 'take_place', 'destination', 'hand_luggage',
       'checked_baggage', 'crawl_date']
df1 = df1[order]

In [75]:
df1['crawl_date'].nunique(), df1['crawl_date'].unique()

(14,
 array(['11-04-2025', '12-04-2025', '13-04-2025', '14-04-2025',
        '15-04-2025', '16-04-2025', '17-04-2025', '23-04-2025',
        '24-04-2025', '25-04-2025', '07-04-2025', '08-04-2025',
        '09-04-2025', '10-04-2025'], dtype=object))

In [76]:
df2['crawl_date'].unique(), df2['crawl_date'].nunique()

(array(['01-05-2025', '02-05-2025', '03-05-2025', '04-05-2025',
        '05-05-2025', '06-05-2025', '07-05-2025', '08-05-2025',
        '09-05-2025', '18-04-2025', '19-04-2025', '20-04-2025',
        '21-04-2025', '22-04-2025', '26-04-2025', '27-04-2025',
        '28-04-2025', '29-04-2025', '30-04-2025'], dtype=object),
 19)

In [77]:
df1.shape, df2.shape

((955, 12), (938, 12))

In [78]:
df = pd.concat([df1, df2], ignore_index=True)
df.shape

(1893, 12)

In [79]:
df.to_csv(f'{city}_merged.csv', index=False)

# PREPROCESSING

In [80]:
city = 'DLI'
df = pd.read_csv(f'{city}_merged.csv')

## Mark id

In [81]:
group_cols = ['brand', 'start_time', 'start_day', 'end_time', 'end_day', 'trip_time']

df['id'] = None

for i, (group_values, group_df) in enumerate(df.groupby(group_cols), start=1):
    group_label = f'{city}{i:04d}' 
    df.loc[group_df.index, 'id'] = group_label

In [82]:
df['id'].nunique()

184

## duplicate

In [83]:
duplicates = df[df.duplicated()]
len(duplicates)

1

In [84]:
df = df.drop_duplicates(keep='first').reset_index(drop=True)
df.shape, df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1892 entries, 0 to 1891
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   brand            1892 non-null   object
 1   price            1892 non-null   object
 2   start_time       1892 non-null   object
 3   start_day        1892 non-null   object
 4   end_time         1892 non-null   object
 5   end_day          1892 non-null   object
 6   trip_time        1892 non-null   object
 7   take_place       1892 non-null   object
 8   destination      1892 non-null   object
 9   hand_luggage     1477 non-null   object
 10  checked_baggage  1477 non-null   object
 11  crawl_date       1892 non-null   object
 12  id               1892 non-null   object
dtypes: object(13)
memory usage: 192.3+ KB


((1892, 13), None)

## brand

In [85]:
df['brand'].nunique(), df['brand'].unique()

(4,
 array(['Bamboo Airways', 'VietJet Air', 'Vietnam Airlines',
        'Bamboo Airways, VietJet Air'], dtype=object))

In [86]:
len(df)

1892

drop 1-stop flights

In [87]:
direct_flights = ['Bamboo Airways', 'VietJet Air', 'Vietnam Airlines', 'Vietravel Airlines']
df = df[df['brand'].isin(direct_flights)]
df['brand'].nunique(), df['brand'].unique()

(3, array(['Bamboo Airways', 'VietJet Air', 'Vietnam Airlines'], dtype=object))

In [88]:
len(df)

1891

In [89]:
df['crawl_date'].nunique(), df['crawl_date'].unique()

(33,
 array(['11-04-2025', '12-04-2025', '13-04-2025', '14-04-2025',
        '15-04-2025', '16-04-2025', '17-04-2025', '23-04-2025',
        '24-04-2025', '25-04-2025', '07-04-2025', '08-04-2025',
        '09-04-2025', '10-04-2025', '01-05-2025', '02-05-2025',
        '03-05-2025', '04-05-2025', '05-05-2025', '06-05-2025',
        '07-05-2025', '08-05-2025', '09-05-2025', '18-04-2025',
        '19-04-2025', '20-04-2025', '21-04-2025', '22-04-2025',
        '26-04-2025', '27-04-2025', '28-04-2025', '29-04-2025',
        '30-04-2025'], dtype=object))

## price

In [90]:
df['price'].head()

0    1.740.927 VND/khách
1    1.704.435 VND/khách
2    1.704.435 VND/khách
3    1.932.066 VND/khách
4    1.939.059 VND/khách
Name: price, dtype: object

In [91]:
# Clean and convert the price column
df['price'] = df['price'].str.extract(r'([\d\.]+)')
df['price'] = df['price'].str.replace('.', '', regex=False).astype(int)
df['price'].head()

0    1740927
1    1704435
2    1704435
3    1932066
4    1939059
Name: price, dtype: int64

## time

In [92]:
df['start_time'].nunique(), df['start_time'].dtype, df['end_time'].nunique(), df['end_time'].dtype

(34, dtype('O'), 35, dtype('O'))

In [93]:
df['start_hour'] = df['start_time'].str.split(':').str[0].astype(int)
df['end_hour'] = df['end_time'].str.split(':').str[0].astype(int)
df[['start_time', 'start_hour', 'end_time', 'end_hour']].head()

,start_time,start_hour,end_time,end_hour
0,16:45,16,17:40,17
1,09:55,9,10:50,10
2,15:35,15,16:30,16
3,21:10,21,22:10,22
4,05:45,5,06:40,6


In [94]:
df['start_hour'] = pd.cut(
    df['start_hour'],
    bins=[0, 3, 9, 15, 21, 24],
    labels=['EarlyMorning', 'Morning', 'Afternoon', 'Evening', 'LateNight'],
    include_lowest=True
)
df['end_hour'] = pd.cut(
    df['end_hour'],
    bins=[0, 3, 9, 15, 21, 24],
    labels=['EarlyMorning', 'Morning', 'Afternoon', 'Evening', 'LateNight'],
    include_lowest=True
)
df[['start_time', 'start_hour', 'end_time', 'end_hour']].head()

,start_time,start_hour,end_time,end_hour
0,16:45,Evening,17:40,Evening
1,09:55,Morning,10:50,Afternoon
2,15:35,Afternoon,16:30,Evening
3,21:10,Evening,22:10,LateNight
4,05:45,Morning,06:40,Morning


In [95]:
time_parts = df['trip_time'].str.extract(r'(?:(?P<hour>\d+)h)?\s*(?:(?P<minute>\d+)m)?')
time_parts = time_parts.astype(float).fillna(0)

df['trip_hour'] = time_parts['hour'] + time_parts['minute'] / 60

df[['trip_time', 'trip_hour']].value_counts()

trip_time  trip_hour
55m        0.916667     1632
1h 0m      1.000000      199
2h 5m      2.083333       29
2h 10m     2.166667       13
1h 20m     1.333333       11
50m        0.833333        4
1h 45m     1.750000        2
1h 25m     1.416667        1
Name: count, dtype: int64

In [96]:
df.drop(columns=['start_time', 'end_time', 'trip_time'], inplace=True)

## day

In [97]:
df['start_day'].unique(), df['start_day'].dtype, df['end_day'].unique(), df['end_day'].dtype

(array(['01 thg 5', '02 thg 5', '03 thg 5', '04 thg 5', '05 thg 5',
        '06 thg 5', '07 thg 5', '08 thg 5', '09 thg 5', '10 thg 5',
        '11 thg 5', '21 thg 4', '22 thg 4', '23 thg 4', '24 thg 4',
        '25 thg 4', '26 thg 4', '27 thg 4', '28 thg 4', '29 thg 4',
        '30 thg 4'], dtype=object),
 dtype('O'),
 array(['01 thg 5', '02 thg 5', '03 thg 5', '04 thg 5', '05 thg 5',
        '06 thg 5', '07 thg 5', '08 thg 5', '09 thg 5', '10 thg 5',
        '11 thg 5', '21 thg 4', '22 thg 4', '23 thg 4', '24 thg 4',
        '25 thg 4', '26 thg 4', '27 thg 4', '28 thg 4', '29 thg 4',
        '30 thg 4'], dtype=object),
 dtype('O'))

In [98]:
df[['start_day', 'end_day']].head()

,start_day,end_day
0,01 thg 5,01 thg 5
1,01 thg 5,01 thg 5
2,01 thg 5,01 thg 5
3,01 thg 5,01 thg 5
4,01 thg 5,01 thg 5


In [99]:
def convert_vn_date(date_str, year=2025):
    day, month = date_str.strip().split(' thg ')
    dt = pd.to_datetime(f"{day}-{int(month):02d}-{year}", dayfirst=True)
    return dt

df['start_day'] = df['start_day'].apply(lambda x: convert_vn_date(x, 2025))
df['end_day'] = df['end_day'].apply(lambda x: convert_vn_date(x, 2025))
df[['start_day', 'end_day']].head(), df['start_day'].dtype, df['end_day'].dtype

(   start_day    end_day
 0 2025-05-01 2025-05-01
 1 2025-05-01 2025-05-01
 2 2025-05-01 2025-05-01
 3 2025-05-01 2025-05-01
 4 2025-05-01 2025-05-01,
 dtype('<M8[ns]'),
 dtype('<M8[ns]'))

In [100]:
holidays = [
    pd.Timestamp('2025-04-30').date(),
    pd.Timestamp('2025-05-01').date(),
]

nearby_holidays = [
    pd.Timestamp('2025-04-29').date(),
    pd.Timestamp('2025-05-02').date(),
    pd.Timestamp('2025-05-03').date(),
    pd.Timestamp('2025-05-04').date(),
]

def is_holiday(date):
    d = date.date()
    if d in holidays:
        return 3
    elif d in nearby_holidays:
        return 2
    elif d.weekday() >= 5:  # Saturday = 5
        return 1
    else:
        return 0
    
df['is_holiday'] = df['start_day'].apply(is_holiday)
df[['start_day', 'is_holiday']].value_counts().head(5)

start_day   is_holiday
2025-04-29  2             121
2025-05-02  2             120
2025-05-03  2             115
2025-05-04  2             110
2025-04-26  1             107
Name: count, dtype: int64

In [101]:
df['crawl_date'] = pd.to_datetime(df['crawl_date'], dayfirst=True).dt.date
df[['start_day', 'crawl_date']].head()

,start_day,crawl_date
0,2025-05-01,2025-04-11
1,2025-05-01,2025-04-11
2,2025-05-01,2025-04-11
3,2025-05-01,2025-04-11
4,2025-05-01,2025-04-11


In [102]:
df['days_left'] = (pd.to_datetime(df['start_day']) - pd.to_datetime(df['crawl_date'])).dt.days
df[['start_day', 'crawl_date', 'days_left']].value_counts().head()

start_day   crawl_date  days_left
2025-04-26  2025-04-23  3            13
2025-05-11  2025-04-23  18           12
2025-05-07  2025-04-23  14           10
            2025-04-28  9            10
2025-05-02  2025-04-19  13           10
Name: count, dtype: int64

In [103]:
df.drop(columns=['start_day', 'end_day', 'crawl_date'], inplace=True)

## Take_place, Destination

In [104]:
df['take_place'].unique(), df['take_place'].dtype

(array(['TP HCM (SGN)\nSân bay Tân Sơn Nhất',
        'TP HCM (SGN)\nSân bay Tân Sơn Nhất\nNhà ga 1',
        'TP HCM (SGN)\nSân bay Tân Sơn Nhất\nNhà ga 3',
        'TP HCM (SGN)\nSân bay Tân Sơn Nhất\nNhà ga T3'], dtype=object),
 dtype('O'))

In [105]:
df.drop('take_place', axis=1, inplace=True)

In [106]:
df['destination'].unique()

array(['Đà Lạt (DLI)\nSân bay Liên Khương',
       'Hà Nội (HAN)\nSân bay Nội Bài\nNhà ga 1',
       'Hà Nội (HAN)\nSân bay Nội Bài', 'Đà Nẵng (DAD)\nSân bay Đà Nẵng',
       'Vinh (VII)\nSân bay Vinh'], dtype=object)

In [107]:
len(df)

1891

In [108]:
valid_destinations  = ['Hà Nội (HAN)\nSân bay Nội Bài', 'Hà Nội (HAN)\nSân bay Nội Bài\nNhà ga 1']
df = df[df['destination'].isin(valid_destinations)]
df['destination'].value_counts()

destination
Hà Nội (HAN)\nSân bay Nội Bài              32
Hà Nội (HAN)\nSân bay Nội Bài\nNhà ga 1    10
Name: count, dtype: int64

In [109]:
len(df)

42

In [110]:
df = df.drop(['destination'], axis=1)

## luggage:

In [111]:
df['hand_luggage_kg'] = df['hand_luggage'].str.extract(r'(\d+)\s*kg').astype(float)
df['checked_baggage_kg'] = df['checked_baggage'].str.extract(r'(\d+)\s*kg').astype(float)
df[['hand_luggage', 'hand_luggage_kg', 'checked_baggage', 'checked_baggage_kg']].value_counts()

hand_luggage                hand_luggage_kg  checked_baggage    checked_baggage_kg
Hành lý xách tay 1 x 12 kg  12.0             Hành lý 23 kg      23.0                  20
                                             Hành lý 1 x 23 kg  23.0                  11
Hành lý xách tay 7 kg       7.0              Hành lý 0 kg       0.0                   10
Name: count, dtype: int64

In [112]:
df['hand_luggage'] = df['hand_luggage_kg']
df['checked_baggage'] = df['checked_baggage_kg']
df.drop(columns=['hand_luggage_kg', 'checked_baggage_kg'], inplace=True)

In [113]:
# df['luggage'] = df['hand_luggage_kg'].fillna(0) + df['checked_baggage_kg'].fillna(0)
# df['luggage'].value_counts()

In [114]:
# df.drop(['checked_baggage', 'hand_luggage', 'checked_baggage_kg', 'hand_luggage_kg'], axis=1, inplace=True)

## organizing

In [115]:
df.columns

Index(['brand', 'price', 'hand_luggage', 'checked_baggage', 'id', 'start_hour',
       'end_hour', 'trip_hour', 'is_holiday', 'days_left'],
      dtype='object')

In [116]:
order = ['id', 'brand', 'price', 'start_hour', 'end_hour', 'trip_hour', 'hand_luggage', 'checked_baggage', 'is_holiday', 'days_left']
df = df[order]
df.head()

,id,brand,price,start_hour,end_hour,trip_hour,hand_luggage,checked_baggage,is_holiday,days_left
245,DLI0027,VietJet Air,3060355,Morning,Morning,2.166667,7.0,0.0,0,14
246,DLI0067,Vietnam Airlines,3766042,Morning,Morning,2.166667,12.0,23.0,0,14
247,DLI0088,Vietnam Airlines,3979522,Morning,Morning,2.083333,12.0,23.0,0,14
248,DLI0101,Vietnam Airlines,4679512,Morning,Afternoon,2.083333,12.0,23.0,0,14
249,DLI0094,Vietnam Airlines,4839106,Morning,Morning,2.083333,12.0,23.0,0,14


In [117]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 42 entries, 245 to 1767
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   id               42 non-null     object  
 1   brand            42 non-null     object  
 2   price            42 non-null     int64   
 3   start_hour       42 non-null     category
 4   end_hour         42 non-null     category
 5   trip_hour        42 non-null     float64 
 6   hand_luggage     41 non-null     float64 
 7   checked_baggage  41 non-null     float64 
 8   is_holiday       42 non-null     int64   
 9   days_left        42 non-null     int64   
dtypes: category(2), float64(3), int64(3), object(2)
memory usage: 3.4+ KB


In [118]:
df.to_csv(f'{city}_preprocessed.csv', index=False)